# Showcase: thesis tournament results

Parses the **shipped tournament CSV** under
`data/tournaments/single_map/` (19 agents x 1 map x 5 iterations =
1710 games) and plots the head-to-head win-rate matrix that backs the
dissertation's headline result.

No Java, no GPU, no fresh games: pure pandas / matplotlib on the
already-shipped data. ~10 seconds end to end on a laptop.

Run from the repo root.

## 1. Load the tournament games

In [ ]:
import json

import pandas as pd

from microrts_agent.paths import PROJECT_ROOT

tournament_json = PROJECT_ROOT / "data" / "tournaments" / "single_map" / "tournament_parsed.json"
with open(tournament_json) as f:
    data = json.load(f)

ais = data["tournament"]["ais"]
games = data["games"]
print(f"AIs in tournament: {len(ais)}")
print(f"Games            : {len(games)}")
print(f"Map              : {data['tournament']['maps'][0]}")
print(f"Iterations       : {data['tournament']['iterations']} per matchup per position")

## 2. Compute the head-to-head win-rate matrix

For every ordered pair `(row, col)`, count games where `row` played
(either as P0 or P1) against `col` and won. Win rate ignores the position
(P0/P1) so the matrix is row's overall WR vs col.

In [ ]:
rows = []
for g in games:
    ai1 = g["players"]["ai1"]["name"]
    ai2 = g["players"]["ai2"]["name"]
    winner = g["result"]["winner"]  # 0 = ai1, 1 = ai2, -1 = draw
    # Record both directions (one row per (perspective, opponent) outcome).
    rows.append(
        {
            "agent": ai1,
            "opponent": ai2,
            "won": 1 if winner == 0 else 0,
            "drew": 1 if winner == -1 else 0,
        }
    )
    rows.append(
        {
            "agent": ai2,
            "opponent": ai1,
            "won": 1 if winner == 1 else 0,
            "drew": 1 if winner == -1 else 0,
        }
    )

df = pd.DataFrame(rows)
agg = (
    df.groupby(["agent", "opponent"])
    .agg(
        wins=("won", "sum"),
        draws=("drew", "sum"),
        games=("won", "count"),
    )
    .reset_index()
)
# Standard scoring: win = 1, draw = 0.5, loss = 0
agg["score"] = (agg["wins"] + 0.5 * agg["draws"]) / agg["games"]

matrix = agg.pivot(index="agent", columns="opponent", values="score")
matrix = matrix.reindex(index=ais, columns=ais)  # canonical order
print("Matrix shape:", matrix.shape)
matrix.head()

## 3. Overall pool win rate per agent

Average of each row (excluding the self-play diagonal): summary of
how strong each agent is across the whole field.

In [ ]:
import numpy as np

diag_mask = ~np.eye(len(ais), dtype=bool)
pool_wr = (matrix.where(diag_mask).mean(axis=1) * 100).sort_values(ascending=False)

print("Pool mean win rate (%):")
for ai, wr in pool_wr.items():
    marker = "  "
    if ai.startswith("UECD"):
        marker = "* "
    print(f"  {marker}{ai:25s} {wr:5.2f}")

## 4. Plot the win-rate matrix as a heatmap

Rows sorted by pool WR (strongest first). Look at the top row: it's
`UECD-Best`, the dissertation's headline agent.

In [ ]:
import matplotlib.pyplot as plt

order = pool_wr.index.tolist()  # strongest -> weakest
sorted_matrix = matrix.reindex(index=order, columns=order)

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(sorted_matrix.values * 100, cmap="RdYlGn", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(len(order)))
ax.set_yticks(range(len(order)))
ax.set_xticklabels(order, rotation=60, ha="right", fontsize=8)
ax.set_yticklabels(order, fontsize=8)
ax.set_xlabel("Opponent")
ax.set_ylabel("Agent (row's WR vs col)")
ax.set_title("Single-map tournament: head-to-head score matrix (win + 0.5 * draw)")

# Annotate every cell.
for i in range(len(order)):
    for j in range(len(order)):
        val = sorted_matrix.values[i, j] * 100
        text_color = "white" if val < 30 or val > 70 else "black"
        ax.text(j, i, f"{val:.0f}", ha="center", va="center", color=text_color, fontsize=6)

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Score (%)")
plt.tight_layout()
plt.show()

## 5. UECD-Best head-to-head detail

Zoom on the headline agent: bar chart of its WR against every other
agent in the field, sorted by difficulty.

In [ ]:
uecd_best_row = matrix.loc["UECD-Best"].drop("UECD-Best").sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = [
    "#d73027" if v < 0.5 else "#fee08b" if v < 0.75 else "#1a9850" for v in uecd_best_row.values
]
ax.barh(range(len(uecd_best_row)), uecd_best_row.values * 100, color=colors)
ax.set_yticks(range(len(uecd_best_row)))
ax.set_yticklabels(uecd_best_row.index, fontsize=9)
ax.set_xlabel("UECD-Best win rate (%) vs opponent")
ax.set_xlim(0, 105)
ax.axvline(50, color="black", linestyle="--", alpha=0.4, linewidth=1)
ax.grid(axis="x", alpha=0.3)
ax.set_title("UECD-Best: head-to-head vs every other agent (single-map tournament)")
for i, v in enumerate(uecd_best_row.values):
    ax.text(v * 100 + 1, i, f"{v * 100:.1f}%", va="center", fontsize=8)
plt.tight_layout()
plt.show()

print(f"UECD-Best vs RAISocketAI: {matrix.loc['UECD-Best', 'RAISocketAI'] * 100:.1f}%")
print(f"UECD-Best overall pool WR : {pool_wr['UECD-Best']:.2f}%")

## Next steps

- Same parse + plot logic against the multi-map tournament: swap the
  path for `data/tournaments/multi_map/tournament_parsed.json`
  (16 agents x 5 maps x 5 iterations = 6000 games).
- For game-theory metrics (Nash, regret, exploitability), see
  `microrts_agent/tournament/ranking/game_theory.py` and
  `data/tournaments/single_map/visualizations/game-theoretic-metrics/`.
- To play a fresh game yourself, see `examples/load_agent.ipynb`.